# VALIDACIÓN CRUZADA.    

La validación cruzada aparece como una forma de aprovechar mejor los datos disponibles para evaluar modelos. En lugar de hacer una única separación, realizamos varias divisiones. Cada parte del dataset tiene oportunidad de funcionar como prueba en algún momento, mientras el resto se usa para entrenar. Todavía no necesitamos entrar en el detalle del procedimiento.     

La idea general es que, en lugar de obtener una única métrica, obtenemos varias métricas, una por cada división. Luego podemos calcular un promedio y observar cuánta variabilidad hay entre resultados. Con una única separación obtenemos una sola métrica. Con validación cruzada obtenemos varias separaciones, varias métricas y una estimación más estable.

La validación cruzada ayuda a comparar con más criterio porque evalúa cada modelo en varias divisiones. Así podemos observar no solo el promedio de desempeño, sino también la variabilidad. Un modelo con promedio apenas mayor pero resultados muy inestables puede no ser preferible a otro un poco más simple y estable.      

Esto no significa que la validación cruzada reemplace completamente al conjunto de prueba final. En muchos flujos de trabajo, se usa validación cruzada para comparar modelos y elegir configuraciones, y se reserva un conjunto de prueba final para una última evaluación honesta. Esa distinción se volverá más importante cuando trabajemos con búsqueda de hiperparámetros.

## K-Fold Cross Validation.   


La validación cruzada permite evaluar un modelo usando varias divisiones de los datos, en lugar de depender de una única separación entre entrenamiento y prueba. La forma más conocida es K-Fold Cross Validation, o validación cruzada con K particiones.    

La idea es dividir el dataset en K partes aproximadamente iguales. A cada una de esas partes se la llama fold. Luego, el modelo se entrena varias veces. En cada repetición, una de las partes se usa como conjunto de validación y las demás se usan para entrenar.      

Por ejemplo, si usamos 5 folds, dividimos los datos en cinco partes: `fold 1`, `fold 2`, `fold 3`, `fold 4` y `fold 5`.    

En la primera repetición, el modelo se entrena con los folds 2, 3, 4 y 5, y se evalúa con el fold 1. En la segunda repetición, se entrena con los folds 1, 3, 4 y 5, y se evalúa
con el fold 2. En la tercera repetición, se entrena con los folds 1, 2, 4 y 5, y se evalúa con el fold 3. Luego se repite el mismo criterio hasta que cada fold haya sido usado una vez como validación.     


Al final, obtenemos cinco resultados de evaluación. Si la métrica es accuracy, tendremos cinco valores de accuracy. Si usamos MAE, tendremos cinco valores de MAE. Si usamos F1-score, tendremos cinco valores de F1-score. Luego podemos calcular el promedio de esos resultados. Ese promedio ofrece una estimación más estable que una única separación train/test, porque cada caso del dataset tuvo oportunidad de formar parte de la validación una vez. Dicho de otra manera, en K-Fold cada parte del dataset se usa una vez para validar y varias veces para entrenar.    


Esto permite aprovechar mejor los datos disponibles. En una única separación train/test, algunos casos quedan reservados para prueba y no participan del entrenamiento. En K-Fold, en cambio, el modelo se entrena varias veces y cada caso se usa para evaluación en alguna repetición.

Es importante entender que *no se obtiene un único modelo durante la validación cruzada*, sino varios entrenamientos temporales. Si usamos 5 folds, el algoritmo entrena 5 modelos, uno por cada división. Estos modelos sirven para evaluar el desempeño esperado. Después, cuando ya elegimos el enfoque, podemos entrenar un modelo final con todos los datos de entrenamiento disponibles.

#### Valor del parámetro **K**      

El valor de K puede variar. Uno de los valores más frecuentes es **K = 5** . También es común usar K = 10 . Si usamos K = 5 , tendremos cinco entrenamientos y cinco evaluaciones. Si usamos K = 10 , tendremos diez entrenamientos y diez evaluaciones.       

En general, si usamos más folds, cada entrenamiento se realiza con una proporción mayor de los datos, pero también se hacen más entrenamientos, por lo que el costo computacional aumenta.


### Características.


K-Fold se usa tanto en **clasificación** como en **regresión**. Lo que cambia es la métrica.

En *clasificación* podemos evaluar `accuracy`, `F1-score`, `recall`, `precision` u otras métricas. En *regresión* podemos evaluar `MAE`, `RMSE`, `R²` u otras. La estructura de la validación es la misma.

Sin embargo, en clasificación hay un detalle importante: si las clases están desbalanceadas, una división K-Fold común puede dejar algunos folds con proporciones de clases poco representativas. Para resolver esto, suele usarse una variante llamada `Stratified K-Fold`, que veremos en la más adelante.

En `scikit-learn` , la validación cruzada puede realizarse con `cross_val_score` . Por ejemplo, para evaluar un modelo con 5 folds:

```python
    from sklearn.model_selection import cross_val_score
    scores = cross_val_score(modelo, X, y, cv=5)
    scores
```
Este código entrena y evalúa el modelo cinco veces, usando distintas particiones. El resultado es una lista de cinco puntajes, uno por cada fold.     

La interpretación dependerá de la métrica usada. Por defecto, `cross_val_score` utiliza la métrica asociada al método score del modelo. En clasificadores, suele ser `accuracy`. En regresores, suele ser `R²`. Si queremos otra métrica, debemos indicarla con el parámetro *scoring*.       

Por ejemplo, para usar `MAE` en regresión, `scikit-learn` emplea una convención particular: muchas métricas de error se expresan como valores negativos dentro de `cross_val_score`, porque la función está pensada para que valores mayores sean mejores. Este detalle puede resultar confuso al principio y conviene tratarlo con cuidado en los notebooks.     

De momento, lo importante no es memorizar todos los parámetros, sino entender el procedimiento: *K-Fold Cross Validation* divide los datos en K partes, entrena K veces, evalúa K veces y obtiene una estimación más estable del desempeño